# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass
from langchain_openai.embeddings import OpenAIEmbeddings
#from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever  # ← CORRECT import for 1.0+


from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
#from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [3]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [5]:
# # https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
# ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# # https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
# ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# # https://platform.openai.com/api-keys
# OPENAI_API_KEY = getpass("OpenAI API key: ")


# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Set a meaningful index name
INDEX_NAME = "ironhack-my-chatbot-index" #give it a meaningful name,

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings  # ✅ Add this line
)


C:\Users\yogan\AppData\Local\Temp\ipykernel_19116\1696196938.py:18: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use `Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [6]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [7]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime # Import datetime here


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    """Populate document metadata from the JSON record.

    Expected (best-effort) keys in `record`:
    - name: str
    - summary: str
    - url: str
    - category: str
    - updated_at: str or epoch (seconds)

    This function is intentionally defensive: it fills what it can and
    normalizes updated_at to ISO 8601 when possible.
    """
    # Best-effort field extraction with reasonable fallbacks
    metadata["name"] = record.get("name") or record.get("title") or record.get("doc_name") or ""
    metadata["summary"] = record.get("summary") or record.get("description") or ""
    metadata["url"] = record.get("url") or record.get("link") or record.get("source") or ""
    metadata["category"] = record.get("category") or record.get("type") or ""

    # Normalize updated_at
    raw_updated = record.get("updated_at") or record.get("updated") or record.get("last_updated") or ""
    iso_updated = ""
    try:
        if isinstance(raw_updated, (int, float)):
            iso_updated = datetime.fromtimestamp(raw_updated).isoformat()
        elif isinstance(raw_updated, str) and raw_updated.strip():
            s = raw_updated.strip()
            # Handle common cases: ISO strings, or "YYYY-MM-DD"
            try:
                iso_updated = datetime.fromisoformat(s.replace("Z", "+00:00")).isoformat()
            except Exception:
                # Try a couple of common formats
                for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y", "%m/%d/%Y"):
                    try:
                        iso_updated = datetime.strptime(s, fmt).isoformat()
                        break
                    except Exception:
                        pass
    except Exception:
        iso_updated = ""

    metadata["updated_at"] = iso_updated

    # Helpful for debugging / attribution in answers
    metadata["source"] = metadata.get("url") or metadata.get("name") or "json_record"
    return metadata



loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [8]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Clean docs metadata
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()

    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()

    # Use model_copy() instead of copy() to avoid Pydantic deprecation warning
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

# Create embeddings ONCE
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create vectorstore ONCE using cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [9]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. Can you provide information on the sales team at NASA?
2. What does the sales team at NASA do?
3. How does NASA's sales team operate?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.360s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

Controlled MultiQuery with structured reformulation + dedup

Goal: force query diversity by categories (definition / org / contact / responsibilities), and deduplicate retrieved docs so you don’t spam the context.

In [10]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

CONTROLLED_MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
You are generating search queries to retrieve internal passages.

Create EXACTLY 4 queries for the user question, one per category:
1) definition / meaning
2) org structure / team ownership
3) contacts / how to reach / role
4) responsibilities / tasks / scope

Return STRICT JSON only, like:
{{
  "queries": [
    {{"category":"definition","query":"..."}},
    {{"category":"org","query":"..."}},
    {{"category":"contacts","query":"..."}},
    {{"category":"responsibilities","query":"..."}}
  ]
}}

User question: {question}
""")

def parse_json_safely(text: str) -> dict:
    # Very small “self-heal” for common JSON mistakes
    text = text.strip()
    # If model wraps in ```json ... ```
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

def dedupe_docs(docs):
    """Deduplicate by (url or name) + page_content prefix"""
    seen = set()
    out = []
    for d in docs:
        key = (
            d.metadata.get("url") or d.metadata.get("name") or d.metadata.get("source"),
            (d.page_content or "")[:200]
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(d)
    return out

def controlled_multiquery_rag(question: str, per_query_k: int = 3, final_k: int = 8):
    query_chain = CONTROLLED_MULTI_QUERY_PROMPT | llm | StrOutputParser()
    raw = query_chain.invoke({"question": question})
    payload = parse_json_safely(raw)

    queries = payload["queries"]

    print("\nStructured Queries:")
    print("------------------")
    for q in queries:
        print(f'- [{q["category"]}] {q["query"]}')
    print("------------------\n")

    all_docs = []
    for q in queries:
        docs = self_healing_retriever(q["query"])
        all_docs.extend(docs[:per_query_k])

    all_docs = dedupe_docs(all_docs)[:final_k]

    # Feed a *clean* combined context into your existing rag_chain
    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs),
    })

print("---- Answer (Iteration 1) ----")
print(controlled_multiquery_rag("what is the nasa sales team?"))

---- Answer (Iteration 1) ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Structured Queries:
------------------
- [definition] Define NASA sales team
- [org] NASA sales team organizational structure
- [contacts] Contact information for NASA sales team
- [responsibilities] Tasks and responsibilities of NASA sales team
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What is the role of the sales team at NASA?', '2. Can you explain the responsibilities of the sales team within NASA?', "3. How would you describe the functions of NASA's sales team?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.329s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.469s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


Iteration 2 (New & creative): “Query expansion → answer → critique → follow-up retrieval → final answer”

Goal: demonstrate that MultiQuery concepts can be used in an iterative retrieval loop (a lightweight “agentic” RAG), not only a single pass.

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

EXPAND_QUERIES_PROMPT = ChatPromptTemplate.from_template("""
Generate 5 search queries to answer the user question well.
Rules:
- queries must be diverse (synonyms, acronyms, related departments, internal naming)
- queries should be short (<= 12 words)
Return one query per line.

Question: {question}
""")

CRITIQUE_PROMPT = ChatPromptTemplate.from_template("""
You are auditing an answer for missing evidence.

Given:
Question: {question}
Draft answer: {draft}

1) List up to 3 missing info gaps that likely require more retrieval.
2) For each gap, propose 1 targeted search query.

Return STRICT JSON:
{{
  "gaps": [
    {{
      "gap": "...",
      "query": "..."
    }}
  ]
}}
""")

def critique_and_refine_rag(question: str, initial_k: int = 2, followup_k: int = 2, final_k: int = 10):
    # Step 1: multi-query expand
    expand_chain = EXPAND_QUERIES_PROMPT | llm | StrOutputParser()
    expanded = expand_chain.invoke({"question": question})
    queries = [q.strip() for q in expanded.split("\n") if q.strip()]

    print("\nExpanded Queries:")
    print("------------------")
    for q in queries:
        print("-", q)
    print("------------------\n")

    # Step 2: initial retrieval
    docs = []
    for q in queries:
        docs.extend(self_healing_retriever(q)[:initial_k])
    docs = dedupe_docs(docs)

    # Step 3: draft answer
    draft = rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(docs[:final_k]),
    })

    # Step 4: critique + targeted followups
    critique_chain = CRITIQUE_PROMPT | llm | StrOutputParser()
    critique_raw = critique_chain.invoke({"question": question, "draft": draft})
    critique = parse_json_safely(critique_raw)

    print("\nCritique Gaps & Follow-up Queries:")
    print("---------------------------------")
    for g in critique.get("gaps", []):
        print(f'- GAP: {g["gap"]}\n  QUERY: {g["query"]}')
    print("---------------------------------\n")

    followup_docs = []
    for g in critique.get("gaps", [])[:3]:
        followup_docs.extend(self_healing_retriever(g["query"])[:followup_k])

    merged = dedupe_docs(docs + followup_docs)[:final_k]

    # Step 5: final answer with improved evidence
    final = rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(merged),
    })

    return {"draft": draft, "final": final}

print("---- Answer (Iteration 2) ----")
out = critique_and_refine_rag("what is the nasa sales team?")
print("\nDRAFT:\n", out["draft"])
print("\nFINAL:\n", out["final"])

---- Answer (Iteration 2) ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Expanded Queries:
------------------
- 1. "NASA commercial team description"
- 2. "NASA business development department details"
- 3. "NASA revenue generation unit overview"
- 4. "NASA marketing and sales division explanation"
- 5. "NASA sales department functions"
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. "What is the description of NASA\'s commercial team?"', '2. "Can you provide information on the commercial team at NASA?"', '3. "How would you describe NASA\'s team focused on commercial activities?"']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.356s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.467s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/e


Critique Gaps & Follow-up Queries:
---------------------------------
- GAP: Missing information on the roles and responsibilities of the NASA sales team.
  QUERY: NASA sales team roles and responsibilities
- GAP: Missing information on the size and composition of the NASA sales team.
  QUERY: NASA sales team size and composition
- GAP: Missing information on the sales targets or achievements of the NASA sales team.
  QUERY: NASA sales team sales targets or achievements
---------------------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What are the roles and responsibilities of the sales team at NASA?', 'What specific duties do members of the NASA sales team have?', 'Can you provide an overview of the sales team roles and responsibilities within NASA?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.335s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://3e0b79052eeb4eea82bff7a161112221.us-central1.gcp.cloud.es.io:443/ironhack-my-chatbot-index/_search?_source_includes=metadata,text [status:200 duration:0.334s]
INFO:httpx:HTTP Request: POST https://ap


DRAFT:
 The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.

FINAL:
 The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.
